<a href="https://colab.research.google.com/github/codeunitdev/natlas-devkit/blob/main/notebooks/N_ATLAS_DevKit_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# N-ATLaS DevKit — Colab demo

Run Nigeria's open language model **N-ATLaS** on a free Colab GPU, then use it three ways:
the **Python SDK**, an **OpenAI-compatible API**, and a **web playground**.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Sign in at huggingface.co and accept the terms on the [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) model page.
3. Create a *read* token at huggingface.co/settings/tokens and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN`.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 1. Install

In [1]:
!pip install -q "natlas-devkit[all] @ git+https://github.com/codeunitdev/natlas-devkit.git" openai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Hugging Face login

In [2]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Load N-ATLaS in 4-bit (about 6 GB of GPU memory; first download takes a few minutes)

In [3]:
from natlas import NAtlas
llm = NAtlas(quantization="4bit")
print("Loaded", llm.model_id)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded NCAIR1/N-ATLaS


## 4. Python SDK — one model, four languages

In [4]:
prompts = {
    "Hausa":   "menene ake nufi da gwagwarmaya",
    "Yoruba":  "Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?",
    "Igbo":    "Kedu ka m ga-esi chebe onwe m pụọ n'ọrịa ịba?",
    "English": "Explain in simple terms how a farmer can save money with a cooperative.",
}
for lang, q in prompts.items():
    print(f"=== {lang} ===\nQ: {q}\nA: {llm.ask(q, max_new_tokens=250)}\n")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== Hausa ===
Q: menene ake nufi da gwagwarmaya
A: Gwagwarmaya na iya haɗawa da fafatawa ko kuma gasa tsakanin mutane don samun wani abu mai iyaka, kamar albarkatu, matsayi, ko kulawar zamantakewa. Wannan na iya faruwa a fannoni daban-daban na rayuwa, kamar kasuwanci, siyasa, ko dangantaka ta sirri, kuma sau da yawa yana samo asali ne daga buƙatar iyawa iyakance, sha'awar mamaye kasuwa, ko son mulki. Gwagwarmayar na iya haifar da sakamako maras kyau idan ba a sarrafa shi yadda ya kamata ba, amma kuma zai iya haifar da ci gaba da sabbin dabaru.

=== Yoruba ===
Q: Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?
A: Daidaitawa daidai yake da kare kanku daga cututtuka. Ga wasu shawarwarin kariya:

1. **Wanke hannayenku akai-akai**: Wanke hannuwanku da sabulu da ruwa mai gishiri na tsawon dakika 20 akalla. Wannan yana taimakawa wajen cire ƙwayoyin cuta da sauran ƙwayoyin cuta.

2. **A guji taba ido**: Idanun ku suna iya kamuwa da cututtukan ido masu saurin kamuwa da su, kuma ta hanyar taba ido,

### Translation and forced reply language

In [5]:
print(llm.translate("Wash your hands with soap before eating.", to="Hausa"))
print(llm.ask("What are the early signs of malaria?", language="Igbo", max_new_tokens=200))

Ku wanke hannayenku da sabulu kafin ku ci abinci.
Ihe mbụ nke ọrịa shuga bụ ihe mgbaàmà dị ka oyi, ahụ ọkụ na mgbu n'ahụ́.


### Streaming

In [6]:
for piece in llm.stream_chat([{"role": "user", "content": "Give me three tips for saving money as a student."}], max_new_tokens=250):
    print(piece, end="", flush=True)

Sure! Here are three tips for saving money as a student:

1. **Budgeting:** Create and stick to a budget that outlines your income and expenses. This will help you identify areas where you can cut back and save more.

2. **Cook Your Own Meals:** Instead of eating out all the time, consider cooking meals at home. It’s not only healthier but also cheaper.

3. **Use Student Discounts:** Many businesses offer discounts to students. Always ask if there is a discount available, and take advantage of any deals or promotions offered specifically to students.

## 5. OpenAI-compatible API

Start the API server in the background (reusing the loaded model), then call it with the **official OpenAI client**.
Any app built for OpenAI switches to N-ATLaS by changing `base_url`.

In [7]:
import threading, time, uvicorn
from natlas.server import create_app

server = uvicorn.Server(uvicorn.Config(create_app(engine=llm), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
resp = client.chat.completions.create(
    model="n-atlas",
    messages=[{"role": "user", "content": "menene ake nufi da gwagwarmaya"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)
print(resp.usage)

Gwagwarmaya na iya haɗawa da fafatawa ko kuma yin gasa don cimma wani buri, samun nasara kan abokin hamayya, ko shawo kan kalubale ko wahalhalu. Ana amfani da ita a fannoni daban-daban na rayuwa, kamar wasanni, kasuwanci, siyasa, ko ci gaban mutum.
CompletionUsage(completion_tokens=100, prompt_tokens=95, total_tokens=195, completion_tokens_details=None, prompt_tokens_details=None)


## 6. Web playground (opens a temporary public link)

In [8]:
from natlas.playground import build_demo
build_demo(engine=llm).queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c761881bfcd3069a47.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
